# 07B Alarm Burden Sensitivity Analysis

This notebook is an additional refinement after Step 7.

It does **not** retrain models and does **not** replace the main result.  
It adds a more realistic alarm-burden analysis.

Why this is needed:

Step 7 selected thresholds mainly using F2 and false alarm episodes per day.  
However, a model can still have long alarm duration even if the number of separate false alarm episodes is below one per day.

This notebook checks whether stricter alarm rules can reduce:

- total alarm time
- false alarm episodes
- false alarm episodes per day

while still trying to detect the failure event.

It adds a persistence rule:

> An alarm is accepted only if the model score remains above the threshold for at least a minimum duration, such as 5, 10, 15, 30 or 60 minutes.

This is a sensitivity analysis for the dissertation discussion.

In [ ]:
# 1. Imports

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

print("Python executable:", sys.executable)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
# 2. Project paths

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_PREDICTIONS_DIR = PROJECT_ROOT / "outputs" / "predictions"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

validation_predictions_path = OUTPUT_PREDICTIONS_DIR / "validation_predictions_all_models.csv"
test_predictions_path = OUTPUT_PREDICTIONS_DIR / "test_predictions_all_models.csv"

print("Project root:", PROJECT_ROOT)
print("Validation predictions:", validation_predictions_path)
print("Test predictions:", test_predictions_path)

In [ ]:
# 3. Load prediction files

if not validation_predictions_path.exists():
    raise FileNotFoundError("validation_predictions_all_models.csv not found. Run Step 6 first.")

if not test_predictions_path.exists():
    raise FileNotFoundError("test_predictions_all_models.csv not found. Run Step 6 first.")

val_pred = pd.read_csv(validation_predictions_path)
test_pred = pd.read_csv(test_predictions_path)

for data in [val_pred, test_pred]:
    data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
    data.dropna(subset=["timestamp"], inplace=True)
    data.sort_values("timestamp", inplace=True)
    data.reset_index(drop=True, inplace=True)

    if "warning_12h_event_id" not in data.columns:
        raise ValueError("warning_12h_event_id column is missing. Rerun Step 6 with split metadata.")

    data["warning_12h_event_id"] = data["warning_12h_event_id"].fillna("None").astype(str)
    data["y_true"] = data["y_true"].astype(int)

score_columns = [col for col in val_pred.columns if col.endswith("_score")]

if not score_columns:
    raise ValueError("No score columns found. Run Step 6 first.")

model_name_map = {
    "logistic_regression_score": "Logistic Regression",
    "random_forest_score": "Random Forest",
    "xgboost_score": "XGBoost",
    "isolation_forest_score": "Isolation Forest"
}

print("Validation shape:", val_pred.shape)
print("Test shape:", test_pred.shape)

print("\nScore columns:")
for col in score_columns:
    print("-", col, "=>", model_name_map.get(col, col))

print("\nValidation warning events:")
print(val_pred["warning_12h_event_id"].value_counts().head(10))

print("\nTest warning events:")
print(test_pred["warning_12h_event_id"].value_counts().head(10))

In [ ]:
# 4. Define documented failure events

failure_events = pd.DataFrame({
    "event_id": ["F1", "F2", "F3", "F4"],
    "failure_type": ["Air leak", "Air leak", "Air leak", "Air leak"],
    "failure_start": [
        "2020-04-18 00:00",
        "2020-05-29 23:30",
        "2020-06-05 10:00",
        "2020-07-15 14:30"
    ],
    "failure_end": [
        "2020-04-18 23:59",
        "2020-05-30 06:00",
        "2020-06-07 14:30",
        "2020-07-15 19:00"
    ]
})

failure_events["failure_start"] = pd.to_datetime(failure_events["failure_start"])
failure_events["failure_end"] = pd.to_datetime(failure_events["failure_end"])
failure_events["warning_12h_start"] = failure_events["failure_start"] - pd.Timedelta(hours=12)
failure_events["warning_12h_end"] = failure_events["failure_start"] - pd.Timedelta(seconds=1)

failure_events

In [ ]:
# 5. Helper functions

MERGE_GAP_MINUTES = 10

def estimate_sample_interval_minutes(data):
    diffs = data["timestamp"].diff().dropna()
    if diffs.empty:
        return 0.0
    median_seconds = diffs.dt.total_seconds().median()
    if pd.isna(median_seconds) or median_seconds <= 0:
        return 0.0
    return median_seconds / 60

def safe_pr_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return average_precision_score(y_true, y_score)
    except Exception:
        return np.nan

def safe_roc_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan

def create_persistent_predictions(data, score_col, threshold, min_duration_minutes):
    working = data[["timestamp", "y_true", "warning_12h_event_id", score_col]].copy()
    working["raw_alarm"] = (working[score_col] >= threshold).astype(int)
    working["persistent_alarm"] = 0

    positives = working[working["raw_alarm"] == 1].copy()

    episode_columns = [
        "episode_id",
        "episode_start",
        "episode_end",
        "duration_minutes",
        "positive_points",
        "overlaps_warning_window",
        "event_ids_overlapped",
        "accepted_by_persistence"
    ]

    if positives.empty:
        return working["persistent_alarm"].values, pd.DataFrame(columns=episode_columns)

    positives = positives.sort_values("timestamp").reset_index()
    positives = positives.rename(columns={"index": "original_index"})

    time_gap = positives["timestamp"].diff()
    new_episode = time_gap.isna() | (time_gap > pd.Timedelta(minutes=MERGE_GAP_MINUTES))
    positives["episode_id"] = new_episode.cumsum()

    sample_interval_minutes = estimate_sample_interval_minutes(data)
    episode_records = []

    for episode_id, group in positives.groupby("episode_id"):
        start = group["timestamp"].min()
        end = group["timestamp"].max()
        duration_minutes = max((end - start).total_seconds() / 60, 0) + sample_interval_minutes

        event_ids = sorted([
            x for x in group["warning_12h_event_id"].dropna().unique().tolist()
            if str(x) != "None"
        ])

        overlaps_warning = len(event_ids) > 0
        accepted = duration_minutes >= min_duration_minutes

        if accepted:
            original_indices = group["original_index"].tolist()
            working.loc[original_indices, "persistent_alarm"] = 1

        episode_records.append({
            "episode_id": int(episode_id),
            "episode_start": start,
            "episode_end": end,
            "duration_minutes": float(duration_minutes),
            "positive_points": int(len(group)),
            "overlaps_warning_window": bool(overlaps_warning),
            "event_ids_overlapped": ", ".join(event_ids) if event_ids else "None",
            "accepted_by_persistence": bool(accepted)
        })

    episodes = pd.DataFrame(episode_records, columns=episode_columns)
    accepted_episodes = episodes[episodes["accepted_by_persistence"] == True].copy()

    return working["persistent_alarm"].values, accepted_episodes

def record_level_metrics(y_true, y_score, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)
    y_pred = np.asarray(y_pred).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "pr_auc": safe_pr_auc(y_true, y_score),
        "roc_auc": safe_roc_auc(y_true, y_score),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    }

def evaluate_event_and_alarm_burden(data, y_pred, episodes, events, model_name, threshold, min_duration_minutes, split_name):
    working = data.copy()
    working["persistent_prediction"] = np.asarray(y_pred).astype(int)

    split_start = working["timestamp"].min()
    split_end = working["timestamp"].max()

    operating_hours = max((split_end - split_start).total_seconds() / 3600, 1e-9)
    operating_days = operating_hours / 24

    events_in_split = sorted([
        x for x in working["warning_12h_event_id"].dropna().unique().tolist()
        if str(x) != "None"
    ])

    event_records = []

    for _, event in events.iterrows():
        event_id = event["event_id"]

        if event_id not in events_in_split:
            continue

        event_alert_rows = working[
            (working["warning_12h_event_id"] == event_id) &
            (working["persistent_prediction"] == 1)
        ].copy()

        detected = len(event_alert_rows) > 0

        if detected:
            first_alert = event_alert_rows["timestamp"].min()
            lead_time_hours = (event["failure_start"] - first_alert).total_seconds() / 3600
        else:
            first_alert = pd.NaT
            lead_time_hours = np.nan

        event_records.append({
            "split": split_name,
            "model": model_name,
            "threshold": threshold,
            "min_duration_minutes": min_duration_minutes,
            "event_id": event_id,
            "failure_start": event["failure_start"],
            "warning_start": event["warning_12h_start"],
            "detected": detected,
            "first_alert_time": first_alert,
            "lead_time_hours": lead_time_hours
        })

    event_detection = pd.DataFrame(event_records)

    detected_events = int(event_detection["detected"].sum()) if not event_detection.empty else 0
    total_events = int(len(event_detection))
    missed_events = total_events - detected_events

    if episodes.empty:
        total_alarm_episodes = 0
        true_alarm_episodes = 0
        false_alarm_episodes = 0
        total_alarm_time_hours = 0.0
        median_alarm_duration_minutes = np.nan
        max_alarm_duration_minutes = np.nan
    else:
        total_alarm_episodes = int(len(episodes))
        true_alarm_episodes = int(episodes["overlaps_warning_window"].sum())
        false_alarm_episodes = total_alarm_episodes - true_alarm_episodes
        total_alarm_time_hours = float(episodes["duration_minutes"].sum() / 60)
        median_alarm_duration_minutes = float(episodes["duration_minutes"].median())
        max_alarm_duration_minutes = float(episodes["duration_minutes"].max())

    false_alarm_episodes_per_day = false_alarm_episodes / operating_days
    total_alarm_time_percentage = total_alarm_time_hours / operating_hours * 100

    detected_leads = event_detection[event_detection["detected"] == True]["lead_time_hours"] if not event_detection.empty else pd.Series(dtype=float)

    summary = {
        "split": split_name,
        "model": model_name,
        "threshold": threshold,
        "min_duration_minutes": min_duration_minutes,
        "events_evaluated": total_events,
        "events_detected": detected_events,
        "events_missed": missed_events,
        "failure_detection_rate": detected_events / total_events if total_events > 0 else np.nan,
        "median_lead_time_hours": detected_leads.median() if len(detected_leads) > 0 else np.nan,
        "min_lead_time_hours": detected_leads.min() if len(detected_leads) > 0 else np.nan,
        "max_lead_time_hours": detected_leads.max() if len(detected_leads) > 0 else np.nan,
        "total_alarm_episodes": total_alarm_episodes,
        "true_alarm_episodes": true_alarm_episodes,
        "false_alarm_episodes": false_alarm_episodes,
        "false_alarm_episodes_per_day": false_alarm_episodes_per_day,
        "total_alarm_time_hours": total_alarm_time_hours,
        "total_alarm_time_percentage": total_alarm_time_percentage,
        "median_alarm_duration_minutes": median_alarm_duration_minutes,
        "max_alarm_duration_minutes": max_alarm_duration_minutes,
        "alert_episode_precision": true_alarm_episodes / total_alarm_episodes if total_alarm_episodes > 0 else np.nan
    }

    return summary, event_detection

In [ ]:
# 6. Validation sensitivity search

thresholds = np.round(np.arange(0.05, 0.951, 0.05), 2)
min_duration_options = [0, 5, 10, 15, 30, 60]

validation_records = []
validation_event_details = []

for score_col in score_columns:
    model_name = model_name_map.get(score_col, score_col.replace("_score", ""))

    for threshold in thresholds:
        for min_duration in min_duration_options:
            y_true = val_pred["y_true"].astype(int).values
            y_score = val_pred[score_col].values

            y_persistent, episodes = create_persistent_predictions(
                val_pred,
                score_col=score_col,
                threshold=threshold,
                min_duration_minutes=min_duration
            )

            rec_metrics = record_level_metrics(y_true, y_score, y_persistent)

            event_summary, event_detection = evaluate_event_and_alarm_burden(
                val_pred,
                y_persistent,
                episodes,
                failure_events,
                model_name,
                threshold,
                min_duration,
                "validation"
            )

            validation_records.append({
                "score_column": score_col,
                **rec_metrics,
                **event_summary
            })

            if not event_detection.empty:
                validation_event_details.append(event_detection)

validation_sensitivity = pd.DataFrame(validation_records)

if validation_event_details:
    validation_event_details = pd.concat(validation_event_details, ignore_index=True)
else:
    validation_event_details = pd.DataFrame()

validation_sensitivity.to_csv(
    OUTPUT_TABLES_DIR / "validation_alarm_burden_sensitivity_search.csv",
    index=False
)

validation_event_details.to_csv(
    OUTPUT_TABLES_DIR / "validation_alarm_burden_event_details.csv",
    index=False
)

validation_sensitivity.head()

In [ ]:
# 7. Select alarm rules from validation

MAX_FALSE_ALARMS_PER_DAY = 1.0
MAX_ALARM_TIME_PERCENTAGE = 20.0

selected_rule_records = []

for model_name, group in validation_sensitivity.groupby("model"):
    group = group.copy()

    constrained = group[
        (group["false_alarm_episodes_per_day"] <= MAX_FALSE_ALARMS_PER_DAY) &
        (group["total_alarm_time_percentage"] <= MAX_ALARM_TIME_PERCENTAGE)
    ].copy()

    if constrained.empty:
        constrained = group.sort_values(
            by=[
                "failure_detection_rate",
                "false_alarm_episodes_per_day",
                "total_alarm_time_percentage",
                "f2",
                "median_lead_time_hours"
            ],
            ascending=[False, True, True, False, False]
        ).head(1).copy()
        selection_note = "No rule met both alarm-burden constraints; selected best available trade-off."
    else:
        constrained = constrained.sort_values(
            by=[
                "failure_detection_rate",
                "f2",
                "median_lead_time_hours",
                "false_alarm_episodes_per_day",
                "total_alarm_time_percentage"
            ],
            ascending=[False, False, False, True, True]
        ).head(1).copy()
        selection_note = "Selected using validation constraints: false alarms/day <= 1 and alarm time <= 20%."

    row = constrained.iloc[0].to_dict()
    row["selection_note"] = selection_note
    selected_rule_records.append(row)

selected_alarm_rules = pd.DataFrame(selected_rule_records)

selected_alarm_rules.to_csv(
    OUTPUT_TABLES_DIR / "selected_alarm_burden_rules_validation.csv",
    index=False
)

selected_alarm_rules[[
    "model",
    "score_column",
    "threshold",
    "min_duration_minutes",
    "f2",
    "failure_detection_rate",
    "median_lead_time_hours",
    "false_alarm_episodes_per_day",
    "total_alarm_time_percentage",
    "selection_note"
]]

In [ ]:
# 8. Select best overall refined rule from validation

best_refined_rule = selected_alarm_rules.sort_values(
    by=[
        "failure_detection_rate",
        "f2",
        "median_lead_time_hours",
        "false_alarm_episodes_per_day",
        "total_alarm_time_percentage"
    ],
    ascending=[False, False, False, True, True]
).head(1).copy()

best_refined_rule.to_csv(
    OUTPUT_TABLES_DIR / "best_refined_alarm_rule_from_validation.csv",
    index=False
)

best_refined_model = best_refined_rule["model"].iloc[0]
best_refined_threshold = float(best_refined_rule["threshold"].iloc[0])
best_refined_duration = float(best_refined_rule["min_duration_minutes"].iloc[0])
best_refined_score_col = best_refined_rule["score_column"].iloc[0]

print("Best refined validation rule:")
print("Model:", best_refined_model)
print("Score column:", best_refined_score_col)
print("Threshold:", best_refined_threshold)
print("Minimum duration:", best_refined_duration, "minutes")

best_refined_rule

In [ ]:
# 9. Evaluate validation-selected refined rules on the test set

test_records = []
test_event_details = []
test_prediction_output = test_pred.copy()

for _, rule in selected_alarm_rules.iterrows():
    model_name = rule["model"]
    score_col = rule["score_column"]
    threshold = float(rule["threshold"])
    min_duration = float(rule["min_duration_minutes"])

    y_true = test_pred["y_true"].astype(int).values
    y_score = test_pred[score_col].values

    y_persistent, episodes = create_persistent_predictions(
        test_pred,
        score_col=score_col,
        threshold=threshold,
        min_duration_minutes=min_duration
    )

    pred_col = f"{model_name.lower().replace(' ', '_')}_refined_alarm_prediction"
    test_prediction_output[pred_col] = y_persistent

    rec_metrics = record_level_metrics(y_true, y_score, y_persistent)

    event_summary, event_detection = evaluate_event_and_alarm_burden(
        test_pred,
        y_persistent,
        episodes,
        failure_events,
        model_name,
        threshold,
        min_duration,
        "test"
    )

    test_records.append({
        "score_column": score_col,
        **rec_metrics,
        **event_summary
    })

    if not event_detection.empty:
        test_event_details.append(event_detection)

test_refined_results = pd.DataFrame(test_records).sort_values(
    by=[
        "failure_detection_rate",
        "f2",
        "median_lead_time_hours",
        "false_alarm_episodes_per_day",
        "total_alarm_time_percentage"
    ],
    ascending=[False, False, False, True, True]
)

if test_event_details:
    test_event_details = pd.concat(test_event_details, ignore_index=True)
else:
    test_event_details = pd.DataFrame()

test_refined_results.to_csv(
    OUTPUT_TABLES_DIR / "test_alarm_burden_sensitivity_results.csv",
    index=False
)

test_event_details.to_csv(
    OUTPUT_TABLES_DIR / "test_alarm_burden_event_details.csv",
    index=False
)

test_prediction_output.to_csv(
    OUTPUT_PREDICTIONS_DIR / "test_predictions_refined_alarm_rules.csv",
    index=False,
    chunksize=100_000
)

test_refined_results

In [ ]:
# 10. Compare original Step 7 result with refined Step 7B result

original_results_path = OUTPUT_TABLES_DIR / "test_event_level_results_selected_thresholds.csv"

if original_results_path.exists():
    original_results = pd.read_csv(original_results_path)
    original_results["approach"] = "Step 7 original threshold rule"
else:
    original_results = pd.DataFrame()
    print("Original Step 7 result file not found. Comparison will use Step 7B only.")

refined_results = test_refined_results.copy()
refined_results["approach"] = "Step 7B refined alarm-burden rule"

comparison_cols = [
    "approach",
    "model",
    "threshold",
    "min_duration_minutes",
    "events_evaluated",
    "events_detected",
    "failure_detection_rate",
    "median_lead_time_hours",
    "false_alarm_episodes_per_day",
    "total_alarm_time_hours",
    "total_alarm_time_percentage",
    "total_alarm_episodes",
    "false_alarm_episodes",
    "alert_episode_precision",
    "precision",
    "recall",
    "f2"
]

for col in comparison_cols:
    if col not in original_results.columns:
        original_results[col] = np.nan
    if col not in refined_results.columns:
        refined_results[col] = np.nan

alarm_burden_comparison = pd.concat(
    [
        original_results[comparison_cols],
        refined_results[comparison_cols]
    ],
    ignore_index=True
)

alarm_burden_comparison.to_csv(
    OUTPUT_TABLES_DIR / "alarm_burden_comparison_original_vs_refined.csv",
    index=False
)

alarm_burden_comparison

In [ ]:
# 11. Plot alarm-burden trade-offs on validation

for model_name, group in validation_sensitivity.groupby("model"):
    plt.figure(figsize=(9, 6))

    scatter = plt.scatter(
        group["total_alarm_time_percentage"],
        group["failure_detection_rate"],
        s=50,
        alpha=0.7
    )

    plt.axvline(MAX_ALARM_TIME_PERCENTAGE, linestyle="--", label="20% alarm-time limit")
    plt.xlabel("Total alarm time percentage")
    plt.ylabel("Validation failure detection rate")
    plt.title(f"Alarm burden sensitivity: {model_name}")
    plt.legend()
    plt.tight_layout()

    safe_name = model_name.lower().replace(" ", "_")
    plt.savefig(
        OUTPUT_FIGURES_DIR / f"validation_alarm_burden_sensitivity_{safe_name}.png",
        dpi=300
    )
    plt.show()

In [ ]:
# 12. Plot original vs refined alarm-time comparison

plot_df = alarm_burden_comparison.dropna(subset=["total_alarm_time_hours"]).copy()

if not plot_df.empty:
    labels = plot_df["approach"] + "\n" + plot_df["model"]
    plt.figure(figsize=(12, 6))
    plt.bar(labels, plot_df["total_alarm_time_hours"])
    plt.ylabel("Total alarm time hours")
    plt.title("Original vs refined alarm burden on the test set")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES_DIR / "test_alarm_burden_original_vs_refined.png", dpi=300)
    plt.show()
else:
    print("No alarm-time data available for plotting.")

In [ ]:
# 13. Final summary and checklist

summary = pd.DataFrame({
    "item": [
        "Purpose",
        "Models evaluated",
        "Thresholds searched",
        "Persistence durations searched",
        "False alarm constraint",
        "Alarm time constraint",
        "Best refined model from validation",
        "Best refined threshold",
        "Best refined persistence duration",
        "Key dissertation interpretation"
    ],
    "value": [
        "Alarm burden sensitivity analysis",
        ", ".join(sorted(validation_sensitivity["model"].unique().tolist())),
        "0.05 to 0.95 in 0.05 intervals",
        ", ".join([str(x) for x in min_duration_options]) + " minutes",
        f"<= {MAX_FALSE_ALARMS_PER_DAY} false alarm episode per day",
        f"<= {MAX_ALARM_TIME_PERCENTAGE}% total alarm time",
        best_refined_model,
        best_refined_threshold,
        f"{best_refined_duration} minutes",
        "The refined analysis checks whether failure detection can be maintained while reducing alarm burden."
    ]
})

summary.to_csv(
    OUTPUT_TABLES_DIR / "alarm_burden_sensitivity_summary.csv",
    index=False
)

checklist = pd.DataFrame({
    "task": [
        "Validation predictions loaded",
        "Test predictions loaded",
        "Threshold and persistence search completed",
        "Validation refined rules selected",
        "Best refined rule selected from validation",
        "Refined rules evaluated on test set",
        "Original vs refined comparison saved",
        "Alarm-burden figures saved"
    ],
    "status": [
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete"
    ]
})

checklist.to_csv(
    OUTPUT_TABLES_DIR / "alarm_burden_sensitivity_completion_checklist.csv",
    index=False
)

summary

In [ ]:
# 14. Final confirmation

print("Step 7B complete.")

print("\nKey saved tables:")
for name in [
    "validation_alarm_burden_sensitivity_search.csv",
    "selected_alarm_burden_rules_validation.csv",
    "best_refined_alarm_rule_from_validation.csv",
    "test_alarm_burden_sensitivity_results.csv",
    "test_alarm_burden_event_details.csv",
    "alarm_burden_comparison_original_vs_refined.csv",
    "alarm_burden_sensitivity_summary.csv"
]:
    print("-", name, (OUTPUT_TABLES_DIR / name).exists())

print("\nKey saved prediction file:")
print("-", OUTPUT_PREDICTIONS_DIR / "test_predictions_refined_alarm_rules.csv")

print("\nBest refined validation rule:")
print("Model:", best_refined_model)
print("Threshold:", best_refined_threshold)
print("Minimum duration:", best_refined_duration)